# Leetcode 743 : Network Delay Time

## Problem
- `times[i] = [u, v, w]` is a **directed** signal path from node `u` to node `v` that takes `w` time.
- A signal starts at node `k`. Find the minimum time for every node to receive it: the **largest shortest-path distance** from `k`.
- Return `-1` if even one node cannot be reached. Node labels are `1..n`.
- LeetCode bounds: `1 <= n <= 100`, `1 <= times.length <= 6000`, and `1 <= w <= 100`; an `int` safely stores the shortest distances.

## Approach: Dijkstra + Min-Heap
- All travel times are non-negative, so Dijkstra's algorithm applies.
- Build an adjacency list `adj[u] = {(v, w), ...}`. Keep the edges directed; do not add a reverse edge.
- `result[x]` is the shortest known time from `k` to node `x`; initialize it to infinity and set `result[k] = 0`.
- The min-heap stores `{time, node}` and always processes the currently cheapest known route first; `greater<pair<int, int>>` makes the priority queue a min-heap (ties are ordered by node).

## Steps and Code Logic
1. Allocate `n + 1` distances and adjacency lists because nodes are 1-indexed; index `0` is unused.
2. Add every `[u, v, wt]` to `adj[u]` only.
3. Push `{0, k}` and set `result[k] = 0`.
4. Pop `{d, node}`. If `d > result[node]`, skip it: a better route was found after this older heap entry was added.
5. For each `{nextnode, wt}`, compute `d + wt`. If it is smaller than `result[nextnode]`, update the distance and push the improved pair.
6. After the heap is empty, check nodes `1..n`. Any `INT_MAX` means unreachable, so return `-1`.
7. Otherwise return the maximum distance among nodes `1..n`; this is when the last node receives the signal.

## Example
`n = 4, k = 2`, `times = [[2,1,1], [2,3,1], [3,4,1]]` gives distances from node 2 of `[1, 0, 1, 2]` for nodes `1..4`; answer: `2`.

## Why It Works / Details to Remember
- Relaxation only replaces a distance when a strictly shorter route is found (`<`), avoiding needless equal-distance heap entries.
- The heap may contain old distances; the stale-entry check ensures only a node's current best distance expands its outgoing edges.
- Dijkstra's greedy ordering is valid because edge weights are non-negative. Negative weights require another algorithm.
- A single node (`n = 1`, `k = 1`) returns `0`; a disconnected graph returns `-1`.
- The answer is **not** the longest route: it is the maximum of the shortest routes from `k`.

## Complexity
- Time: `O(V + E log E)` for this lazy min-heap implementation; each edge is examined once and successful relaxations may add heap entries.
- Space: `O(V + E)` for the adjacency list, distances, and heap.
- `V = n`; `E = times.size()`.

In [ ]:
class Solution {
public:
    int networkDelayTime(vector<vector<int>>& times, int n, int k) {
        priority_queue<pair<int, int>, vector<pair<int, int>>,
                       greater<pair<int, int>>>
            pq;
        vector<int> result(n + 1, INT_MAX);

        vector<vector<pair<int, int>>> adj(n + 1);
        for (auto& edges : times) {
            int u = edges[0];
            int v = edges[1];
            int wt = edges[2];
            adj[u].push_back({v, wt});
        }
        pq.push({0, k});
        result[k] = 0;

        while (!pq.empty()) {
            auto [d, node] = pq.top();
            pq.pop();

            if (d > result[node]) {
                continue;
            }
            for (auto& [nextnode, wt] : adj[node]) {
                if (d + wt < result[nextnode]) {
                    result[nextnode] = d + wt;

                    pq.push({d + wt, nextnode});
                }
            }
        }
        for (int i = 1; i <= n; i++) { //1-indexed
            if (result[i] == INT_MAX)
                return {-1};
        }
        int mx = *max_element(result.begin()+1, result.end());
        return mx; // Time: O(V + E log E), Space: O(V + E)
    }
};